# Washington State Electric Vehicle Population Analysis

## Project Overview
This project provides a data-driven dashboard for a regional electrical utility company to plan infrastructure upgrades. By analyzing EV registration density and vehicle types, the utility can proactively manage grid load and strategically place public charging stations.

### Business Objectives:
1. **Identify High-Growth Clusters:** Locate zip codes with high EV density using geographic mapping.
2. **Infrastructure Planning:** Segment vehicles by battery type (BEV vs. PHEV) and CAFV eligibility to estimate power demand.
3. **Market Sentiment:** Track brand dominance to understand consumer trends in the regional market.

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '../datasets/raw_dataset/Electric_Vehicle_Population_Data.csv'
df = pd.read_csv(file_path)

# Quick overview
display(df.head())
print(df.info())

## ETL & Data Cleaning
To make this data actionable, we need to:
1. **Parse Spatial Data:** Extract Latitude and Longitude from the `Vehicle Location` column.
2. **Binary Transformation:** Simplify `Clean Alternative Fuel Vehicle (CAFV) Eligibility` into a binary flag.
3. **Temporal Processing:** Ensure the Model Year is treated correctly for growth analysis.

In [ ]:
# 1. Clean Vehicle Location (POINT (Long Lat))
def extract_coords(location):
    if pd.isna(location):
        return None, None
    # Remove 'POINT (' and ')' then split
    clean_loc = location.replace('POINT (', '').replace(')', '')
    long, lat = clean_loc.split(' ')
    return float(lat), float(long)

df['Latitude'], df['Longitude'] = zip(*df['Vehicle Location'].apply(extract_coords))

# 2. Binary CAFV Eligibility Flag
# 'Clean Alternative Fuel Vehicle Eligible' -> 1, others -> 0
df['CAFV_Eligible'] = df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].apply(
    lambda x: 1 if 'Clean Alternative Fuel Vehicle Eligible' in str(x) else 0
)

# 3. Filter for Washington State (focusing on the utility's regional scope)
df_wa = df[df['State'] == 'WA'].copy()

# Using 'Postal Code' as it is the actual column name in the dataset
display(df_wa[['Postal Code', 'Latitude', 'Longitude', 'CAFV_Eligible']].head())

## Exporting Cleaned Data for Tableau
To build the dashboard in Tableau, we will export the processed `df_wa` DataFrame. This file includes the cleaned Latitude, Longitude, and CAFV eligibility metrics.

In [ ]:
import os

# Export cleaned data to CSV for Tableau
export_dir = '../datasets/cleaned_dataset'
os.makedirs(export_dir, exist_ok=True)
export_path = f'{export_dir}/Cleaned_EV_Population_Data.csv'
df_wa.to_csv(export_path, index=False)

print(f"Success! Cleaned data exported to: {export_path}")